# 05 — Winter Adequacy
Scenario modeling of EU gas storage depletion over the winter heating season (Nov 1 – Apr 1).

In [ ]:
import sys, os, warnings
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
warnings.filterwarnings('ignore')

for _c in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (_c / 'src' / 'agsi_client').exists():
        sys.path.insert(0, str(_c))
        os.chdir(_c)
        print(f"Project root: {_c}")
        break

DATA = Path("data/processed")
df = pd.read_parquet(DATA / "eu_aggregate_full.parquet")
df.index = pd.to_datetime(df.index).tz_localize(None)
df = df.sort_index()
df["year"]  = df.index.year
df["month"] = df.index.month

CAPACITY_TWH   = float(df["workingGasVolume"].dropna().iloc[-1])
CRITICAL_FILL  = 0.20
latest_date    = df["full"].dropna().index[-1]
latest_year    = int(latest_date.year)

print(f"eu_aggregate_full.parquet: {df.shape}")
print(f"  Latest date: {latest_date.date()}")
print(f"  Capacity:    {CAPACITY_TWH:.1f} TWh")


## 1. Starting Level and Demand Scenarios
Establish Nov 1 starting storage and four demand withdrawal rate scenarios.

In [ ]:
# ── Starting Nov 1 fill level ─────────────────────────────────────────────────
# Use actual Nov 1 value if available; otherwise project from current fill
nov1_row = df[
    (df.index.month == 11) & (df.index.day == 1) & (df["year"] == latest_year)
]
if not nov1_row.empty:
    start_fill_pct = float(nov1_row["full"].iloc[-1])
    start_source   = f"actual Nov 1 {latest_year}"
else:
    # Project forward using 30-day average injection rate
    current_fill  = float(df["full"].dropna().iloc[-1])
    current_twh   = float(df["gasInStorage"].dropna().iloc[-1])
    days_to_nov1  = max((pd.Timestamp(latest_year, 11, 1) - latest_date).days, 0)
    inj_rate_gwh  = float(df["injection"].dropna().iloc[-30:].mean())
    proj_twh      = min(current_twh + inj_rate_gwh * days_to_nov1 / 1000, CAPACITY_TWH)
    start_fill_pct = proj_twh / CAPACITY_TWH * 100
    start_source   = f"projected from {latest_date.date()} (30d avg injection)"

start_twh = CAPACITY_TWH * (start_fill_pct / 100)

# ── Demand scenarios: daily withdrawal rates (GWh/day) ────────────────────────
winter_df = df[(df["month"].isin([11,12,1,2,3])) & (df["withdrawal"] > 0)]["withdrawal"]
p50 = float(winter_df.quantile(0.50))
p90 = float(winter_df.quantile(0.90))

DEMAND_SCENARIOS = {
    "Mild   (P10 hist)":    float(winter_df.quantile(0.10)),
    "Normal (P50 hist)":    p50,
    "Cold   (P90 hist)":    p90,
    "Extreme (1.35x P90)":  p90 * 1.35,
}

print(f"Starting Nov 1 storage: {start_twh:.1f} TWh  ({start_fill_pct:.1f}%)  [{start_source}]")
print(f"Working capacity:        {CAPACITY_TWH:.1f} TWh")
print(f"Critical level (20%):    {CAPACITY_TWH * CRITICAL_FILL:.1f} TWh")
print()
print("Demand scenarios (daily withdrawal rate):")
for label, rate in DEMAND_SCENARIOS.items():
    print(f"  {label:<26}: {rate:>6,.0f} GWh/day")


## 2. Winter Depletion Curves
Simulate storage fill % from Nov 1 to Apr 1 under each demand scenario.

In [ ]:
WINTER_START = pd.Timestamp(latest_year, 11, 1)
WINTER_END   = pd.Timestamp(latest_year + 1, 4, 1)
dates = pd.date_range(WINTER_START, WINTER_END, freq="D")

colors = ["#27AE60", "#2E75B6", "#E67E22", "#E74C3C"]

fig = go.Figure()
for (label, rate), color in zip(DEMAND_SCENARIOS.items(), colors):
    storage = [start_twh]
    for _ in dates[1:]:
        storage.append(max(storage[-1] - rate / 1000, 0.0))  # GWh/day -> TWh
    fill_pct = [s / CAPACITY_TWH * 100 for s in storage]
    fig.add_trace(go.Scatter(
        x=dates, y=fill_pct,
        name=label.strip(),
        line=dict(color=color, width=2.2)
    ))

fig.add_hline(y=20, line_dash="dot", line_color="red", line_width=1.5,
              annotation_text="20% Critical")
fig.add_hline(y=30, line_dash="dot", line_color="orange", line_width=1,
              annotation_text="30% Low")
fig.update_layout(
    title=f"EU Gas Storage Winter Depletion Scenarios — {latest_year}/{latest_year + 1}",
    xaxis_title="Date", yaxis_title="Fill (%)",
    yaxis=dict(range=[0, 105]),
    height=490, template="plotly_white"
)
fig.show()


## 3. Days Until 20% Critical Level
For each scenario, identify when — if ever — storage falls below the 20% critical threshold.

In [ ]:
results = []
for label, rate in DEMAND_SCENARIOS.items():
    storage = start_twh
    critical_day = None
    for i, d in enumerate(dates):
        if storage / CAPACITY_TWH <= CRITICAL_FILL:
            critical_day = i
            break
        storage = max(storage - rate / 1000, 0.0)

    apr1_twh  = max(start_twh - rate / 1000 * len(dates), 0.0)
    apr1_fill = apr1_twh / CAPACITY_TWH * 100

    if critical_day is not None:
        crit_date = (WINTER_START + pd.Timedelta(days=critical_day)).strftime("%b %d %Y")
        survives  = False
    else:
        crit_date = "—"
        survives  = True

    results.append({
        "label":        label.strip(),
        "rate":         rate,
        "critical_day": critical_day,
        "crit_date":    crit_date,
        "apr1_fill":    apr1_fill,
        "survives":     survives,
    })

print(f"Winter {latest_year}/{latest_year + 1} — Days Until 20% Critical Level")
print(f"Starting level: {start_fill_pct:.1f}% ({start_twh:.1f} TWh)")
print("=" * 68)
print(f"  {'Scenario':<26} {'Rate':>8} {'Critical':>10} {'By date':>12} {'Apr 1 Fill':>10}")
print(f"  {'-'*64}")
for r in results:
    cd = f"{r['critical_day']}d" if r['critical_day'] else ">151d"
    print(f"  {r['label']:<26} {r['rate']:>7,.0f}  {cd:>10} {r['crit_date']:>12} {r['apr1_fill']:>9.1f}%")


## 4. Adequacy Summary
Which scenarios survive the winter without hitting the critical 20% level?

In [ ]:
print()
print(f"{'='*60}")
print(f"  EU WINTER ADEQUACY SUMMARY  {latest_year}/{latest_year + 1}")
print(f"  Nov 1 start: {start_fill_pct:.1f}%  |  Capacity: {CAPACITY_TWH:.1f} TWh")
print(f"{'='*60}")
print(f"  {'Scenario':<30} {'Hits 20%?':^12} {'Apr 1 Fill':>10}")
print(f"  {'-'*54}")
for r in results:
    status = "YES - RISK" if not r["survives"] else "No - safe"
    print(f"  {r['label']:<30} {status:^12} {r['apr1_fill']:>9.1f}%")
print(f"{'='*60}")

safe_count = sum(1 for r in results if r["survives"])
print(f"  {safe_count}/{len(results)} scenarios survive winter without hitting 20% critical level")
